### Step 7: Download PDFs of the Deeds

Using the documents ID from your neighborhood ACRIS dataset you can access PDFs of the deed via a link like this

https://a836-acris.nyc.gov/DS/DocumentSearch/DocumentImageView?doc_id=FT_3860000959686

This is 100% a playwright situation

Write a playwright script that goes to the page, downloads the complete PDF and name it consistently so that it has the document ID in the name!

In [23]:
import pandas as pd
df = pd.read_csv('data/pt_2.csv')

In [24]:
document_ids=df['document_id'].astype(str).str.strip().unique().tolist()

In [25]:
document_ids

['2016041301502003',
 '2014031801081014',
 '2026041500960001',
 '2026031000969002',
 '2026022000160004',
 '2026031000969001',
 '2026022000160002',
 '2026031000969003',
 '2026030500602001',
 '2026022000160001',
 '2026030500624001',
 '2026030401228001',
 '2026022000160005',
 '2025030200051001',
 '2025031300288001',
 '2025031301018001',
 '2025032500938001',
 '2025021900242001',
 '2024042600150001',
 '2024052300256004',
 '2024042600150006',
 '2024042600150010',
 '2024042600150008',
 '2024042600150003',
 '2024052300256002',
 '2024042600150004',
 '2024042600150007',
 '2024042600150009',
 '2024042600150005',
 '2024052300256003',
 '2024052300256005',
 '2023063000420001',
 '2023061200582001',
 '2023061500925003',
 '2022092900907001',
 '2022102100773001',
 '2022102700843001',
 '2022102700532001',
 '2022100400759003',
 '2022100400759006',
 '2022100301201001',
 '2022092900907002',
 'FT_1410005200241',
 '2022090900540001',
 '2022083000843001',
 '2021072000470001',
 '2021071200713001',
 '20210609006

In [26]:
len(document_ids)

4710

In [28]:
import asyncio
from pathlib import Path
from playwright.async_api import async_playwright, TimeoutError

DOWNLOAD_DIR = Path("downloads")
DOWNLOAD_DIR.mkdir(exist_ok=True)


async def download_doc(page, doc_id: str):

    out_path = DOWNLOAD_DIR / f"{doc_id}.pdf"

    if out_path.exists():
        print(f"Skipping {doc_id}")
        return

    url = f"https://a836-acris.nyc.gov/DS/DocumentSearch/DocumentImageView?doc_id={doc_id}"

    try:
        print(f"Processing {doc_id}")

        await page.goto(url, wait_until="domcontentloaded")

        # IMPORTANT: iframe is the whole app
        frame = page.frame_locator('iframe[name="mainframe"]')

        save_btn = frame.locator('img[title="Save"]')
        await save_btn.wait_for(timeout=60000)

        async with page.expect_download(timeout=60000) as download_info:

            # 1. Click SAVE (inside iframe)
            await save_btn.click()

            # 2. Wait for modal dialog (inside iframe)
            dialog = frame.locator("div.vtm_dialog.vtm_printbox")
            await dialog.wait_for(state="visible", timeout=60000)

            # 3. Click OK inside dialog
            ok_btn = dialog.locator('span.vtmBtn', has_text="OK")
            await ok_btn.wait_for(state="visible", timeout=60000)

            await ok_btn.click()

        download = await download_info.value
        await download.save_as(str(out_path))

        print(f"✓ Saved {doc_id}")

    except TimeoutError:
        print(f"⏱ Timeout: {doc_id}")

    except Exception as e:
        print(f"✗ Error {doc_id}: {e}")


async def run(document_ids):

    async with async_playwright() as p:

        browser = await p.chromium.launch(headless=False)

        context = await browser.new_context(accept_downloads=True)

        page = await context.new_page()

        for doc_id in document_ids:
            await download_doc(page, doc_id)

        await browser.close()


# RUN IN NOTEBOOK:
await run(document_ids)

Skipping 2016041301502003
Skipping 2014031801081014
Processing 2026041500960001


CancelledError: 

### Step 8: Upload the Deeds to your investigation!

And let's see how my instance of openaleph holds up!!! (Not wonderfully so far...)

I recommend just using the UI to upload, but here is an option via python

In [ ]:
#You could do this--or just use the interface
# from pathlib import Path
# import time
# COLLECTION_ID = '19'

# for doc in document_ids:
#     file_path = Path('acris_pdfs_flatbush/ACRIS_DEED_'+doc+'.pdf')
#     if file_path.exists():
#         size_in_mb = file_path.stat().st_size / (1024 * 1024)
#         if size_in_mb < 1:
#             print(f"Uploading: {file_path.name}...")
#             try:
#                 result=api.ingest_upload(
#                     collection_id=COLLECTION_ID, 
#                     file_path=file_path,  
#                     metadata={
#                         'file_name': file_path.name,
#                         'title': file_path.stem
#                     }
#                 )
#                 pdf_id_map[doc] = result.get('id')
#                 print(f"  → entity ID: {result.get('id')}")
#                 time.sleep(4)
#             except Exception as e:
#                 print(f"Failed to upload {file_path.name}: {e}")
#     else:
#         pdf_id_map[doc] = 'NotUploaded'
#         print(doc + ' no file')


# print("Batch upload complete!")

### Step 9: Retrive the openaleph document ids from your investigation and merge with your ACRIS df

This is a quirk of openaleph that I actually find quite odd, but you need to use the internal, aleph-generated document ID that is produced after your upload them, if you want to connect your documents to the entities.

Below I use **requests** to access the api.

In [ ]:
import os
import requests
ALEPH_KEY = os.environ.get("OPAL_API_KEY")


In [ ]:


HOST = "https://aleph.floatingmedia.com"
COLLECTION_ID='19'

headers = {"Authorization": f"ApiKey {ALEPH_KEY}"}

response = requests.get(
    f"{HOST}/api/2/entities",
    headers=headers,
    params={"filter:collection_id": COLLECTION_ID, "filter:schema": "Pages",
               "limit": 50,
    "offset": 0}
)
print(response.status_code)


In [ ]:
print(response.json())

In [ ]:
pdf_id_map = {}
num=1
docs_info = response.json()
for doc in docs_info['results']:
    print(doc['id'])
    doc_id = doc['properties']['fileName'][0].split("DEED_")[-1].replace(".pdf","")
    print(doc_id)
    print(num)
    pdf_id_map[int(doc_id)] = doc['id']
    num+=1

In [ ]:
pdf_id_map

In [ ]:
df_acris_final

In [ ]:
df_acris_final['pdf_entity_id'] = df_acris_final['document_id'].map(pdf_id_map)

In [ ]:
df_acris_final

In [ ]:
df_acris_final.to_csv('acris_doc_ids.csv', index=False)

### Step 10: map to ftm using an version of acris.yml


In [32]:
df.columns

Index(['corporationname', 'building_address', 'business_address',
       'registrationid', 'buildingid', 'boro', 'block', 'lot', 'bin',
       'registrationcontactid', 'document_id', 'record_type', 'borough',
       'block_1', 'lot_1', 'easement', 'partial_lot', 'air_rights',
       'subterranean_rights', 'property_type', 'street_number', 'street_name',
       'unit', 'good_through_date', 'document_id_1', 'record_type_1',
       'party_type', 'name', 'address_1', 'address_2', 'country', 'city',
       'state', 'zip', 'good_through_date_1'],
      dtype='object')

In [35]:
df_test = df[['boro', 'block', 'lot']].drop_duplicates()

In [36]:
df_test.shape

(224, 3)